# FACUNDO - Best and worst boxers

## Antes de ejecutar
Este cuaderno contiene solo la parte de Facundo. Necesita el archivo `boxers.db` creado y rellenado por el cuaderno del grupo. En Colab, sube `boxers.db` desde el panel Archivos.

## Criterios
- `win_rate`: victorias / combates totales * 100.
- `points`: 3 puntos por victoria y 1 por empate.

Se muestran todos los empatados. Los boxeadores sin combates no se comparan. La función consulta los datos y no los modifica.


In [1]:
import sqlite3
import os


In [2]:
def show_best_and_worst_boxer(metric: str = "win_rate") -> None:
  """
  Print the best and worst boxers.

  metric: "win_rate" for win percentage, or "points" for
  3 points per win and 1 per draw.
  """
  if metric != "win_rate" and metric != "points":
    print('Choose "win_rate" or "points".')
    return

  if not os.path.isfile("boxers.db"):
    print("Upload boxers.db before running this function.")
    return

  connection = sqlite3.connect("boxers.db")

  try:
    cursor = connection.cursor()
    cursor.execute("SELECT name, total_fights, wins, loses, draws FROM boxers")
    boxers = cursor.fetchall()
  except sqlite3.Error as error:
    print("Error reading the database:", error)
    return
  finally:
    connection.close()

  best_score = None
  worst_score = None
  best_boxers = []
  worst_boxers = []

  for boxer in boxers:
    name, total_fights, wins, loses, draws = boxer

    if total_fights > 0:
      if metric == "win_rate":
        score = wins / total_fights * 100
      else:
        score = wins * 3 + draws

      info = f"{name}: {total_fights} fights, {wins} wins, {loses} losses, {draws} draws"

      if best_score is None or score > best_score:
        best_score = score
        best_boxers = [info]
      elif score == best_score:
        best_boxers.append(info)

      if worst_score is None or score < worst_score:
        worst_score = score
        worst_boxers = [info]
      elif score == worst_score:
        worst_boxers.append(info)

  if len(best_boxers) == 0:
    print("There are no boxers with fights to compare.")
    return

  print("Best boxer(s) by", metric, "- Score:", round(best_score, 2))
  for boxer in best_boxers:
    print(boxer)

  print()
  print("Worst boxer(s) by", metric, "- Score:", round(worst_score, 2))
  for boxer in worst_boxers:
    print(boxer)



## Ejemplos

Ejecuta las dos celdas siguientes para comparar los criterios.


In [3]:
show_best_and_worst_boxer()


Best boxer(s) by win_rate - Score: 100.0
Floyd Mayweather Jr.: 50 fights, 50 wins, 0 losses, 0 draws
Oleksander Usyk: 25 fights, 25 wins, 0 losses, 0 draws

Worst boxer(s) by win_rate - Score: 77.19
Evander Holyfield: 57 fights, 44 wins, 10 losses, 2 draws


In [4]:
show_best_and_worst_boxer("points")


Best boxer(s) by points - Score: 323
Julio Cesar Chavez: 115 fights, 107 wins, 6 losses, 2 draws

Worst boxer(s) by points - Score: 75
Oleksander Usyk: 25 fights, 25 wins, 0 losses, 0 draws


## Cómo explicar la función

1. Comprobamos que el criterio es válido y que existe `boxers.db`.
2. `SELECT` lee los datos y `fetchall()` los guarda en una lista. Después cerramos la conexión. `except` muestra los errores y `finally` cierra la conexión incluso si hay un error.
3. Empezamos con las puntuaciónes en `None` porque todavía no hemos comparado a nadie. Las listas de mejores y peores empiezan vacías.
4. El bucle `for` recorre los boxeadores. Repartimos los datos de cada registro entre cinco variables.
5. Si tiene combates, calculamos su puntuación segun el criterio elegido.
6. Si supera la mejor puntuación, sustituimos la lista de mejores por ese boxeador. Si empata, lo añadimos con `append()`. Repetimos el proceso con la peor puntuación.
7. Mostramos las puntuaciónes con dos decimales y recorremos las listas para imprimir los boxeadores.

**Para contarlo en clase:** Recorro los boxeadores uno a uno, calculo su puntuación y la comparo con la mejor y la peor que llevo guardadas. Al final muestro las dos listas.
